# 🐦 Conteo de Aves — Formato Pivotado

Salida: una fila por **Foto + Clase**, columnas por cada **Modelo_Confianza**.

In [1]:
# ============================================================
# 1. CONFIGURA TUS RUTAS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
# ←←← AJUSTA AQUÍ ←←←
RUTA_FOTOS   = Path('/content/drive/MyDrive/conteo de aves marinas/fotos')
RUTA_MODELOS = Path('/content/drive/MyDrive/conteo de aves marinas/modelos')
RUTA_SALIDA  = Path('/content/drive/MyDrive/conteo de aves marinas/conteos_pivotado.xlsx')

TAMANOS      = ['n', 's', 'm', 'l', 'x']
#CONFIDENCIAS = [0.25, 0.7]
CONFIDENCIAS = np.arange(0.20, 0.91, 0.01).round(2).tolist()

print('Fotos  :', RUTA_FOTOS,   '✅' if RUTA_FOTOS.exists() else '❌')
print('Modelos:', RUTA_MODELOS, '✅' if RUTA_MODELOS.exists() else '❌')

Mounted at /content/drive
Fotos  : /content/drive/MyDrive/conteo de aves marinas/fotos ✅
Modelos: /content/drive/MyDrive/conteo de aves marinas/modelos ✅


In [2]:
CONFIDENCIAS

[0.2,
 0.21,
 0.22,
 0.23,
 0.24,
 0.25,
 0.26,
 0.27,
 0.28,
 0.29,
 0.3,
 0.31,
 0.32,
 0.33,
 0.34,
 0.35,
 0.36,
 0.37,
 0.38,
 0.39,
 0.4,
 0.41,
 0.42,
 0.43,
 0.44,
 0.45,
 0.46,
 0.47,
 0.48,
 0.49,
 0.5,
 0.51,
 0.52,
 0.53,
 0.54,
 0.55,
 0.56,
 0.57,
 0.58,
 0.59,
 0.6,
 0.61,
 0.62,
 0.63,
 0.64,
 0.65,
 0.66,
 0.67,
 0.68,
 0.69,
 0.7,
 0.71,
 0.72,
 0.73,
 0.74,
 0.75,
 0.76,
 0.77,
 0.78,
 0.79,
 0.8,
 0.81,
 0.82,
 0.83,
 0.84,
 0.85,
 0.86,
 0.87,
 0.88,
 0.89,
 0.9]

In [3]:
# ============================================================
# 2. INSTALAR Y CARGAR
# ============================================================

!pip install ultralytics openpyxl -q

from ultralytics import YOLO
from collections import defaultdict
import pandas as pd
from pathlib import Path
import yaml

with open(RUTA_MODELOS / 'data.yaml', 'r') as f:
    data = yaml.safe_load(f)
names = data['names']
CLASES = names if isinstance(names, list) else [names[i] for i in sorted(names, key=int)]
print('Clases:', CLASES)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Clases: ['chuita', 'chuita adulta', 'cushuri adulto', 'cushuri juvenil', 'gallinazo cabeza roja', 'gaviota peruana adulta', 'guanay adulto', 'pelicano adulto', 'pelicano juvenil', 'pichon pinguino', 'pichon piquero', 'pinguino adulto', 'pinguino juvenil', 'piquero adulto', 'piquero juvenil', 'zarcillo']


In [4]:
# ============================================================
# 3. LISTAR FOTOS
# ============================================================

EXTS = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
fotos = sorted([f for f in RUTA_FOTOS.iterdir() if f.suffix in EXTS])
print(f'Total fotos: {len(fotos)}')
for f in fotos[:5]:
    print(' ', f.name)
if len(fotos) > 5:
    print(f'  ... y {len(fotos)-5} más')

Total fotos: 115
  371.jpg
  378.jpg
  380.jpg
  384.jpg
  389.jpg
  ... y 110 más


In [5]:
# ============================================================
# 4. PREDECIR Y ACUMULAR EN FORMATO LONG (imagen por imagen)
# ============================================================

filas_long = []   # cada fila: {FOTO, Clase, modelo_conf, Conteo}

for tam in TAMANOS:
    ruta_modelo = RUTA_MODELOS / f'26{tam}.pt'
    if not ruta_modelo.exists():
        print(f'⚠️  No existe: {ruta_modelo}')
        continue

    print(f'\n🔄 Cargando 26{tam}...')
    modelo = YOLO(str(ruta_modelo))

    for conf in CONFIDENCIAS:
        col_name = f'{tam}_{conf:.2f}'
        print(f'   {col_name} ...', end=' ')

        for ruta_foto in fotos:
            foto = ruta_foto.stem

            r = modelo.predict(
                source=str(ruta_foto),
                conf=conf,
                imgsz=640,
                rect=True,
                verbose=False
            )[0]

            conteo = defaultdict(int)
            if r.boxes:
                for box in r.boxes:
                    conteo[CLASES[int(box.cls)]] += 1

            for clase in CLASES:
                filas_long.append({
                    'FOTO': foto,
                    'Clase': clase,
                    'modelo_conf': col_name,
                    'Conteo': conteo.get(clase, 0)
                })

        print('OK')

print(f'\n✅ Total filas long: {len(filas_long)}')


🔄 Cargando 26n...
   n_0.20 ... OK
   n_0.21 ... OK
   n_0.22 ... OK
   n_0.23 ... OK
   n_0.24 ... OK
   n_0.25 ... OK
   n_0.26 ... OK
   n_0.27 ... OK
   n_0.28 ... OK
   n_0.29 ... OK
   n_0.30 ... OK
   n_0.31 ... OK
   n_0.32 ... OK
   n_0.33 ... OK
   n_0.34 ... OK
   n_0.35 ... OK
   n_0.36 ... OK
   n_0.37 ... OK
   n_0.38 ... OK
   n_0.39 ... OK
   n_0.40 ... OK
   n_0.41 ... OK
   n_0.42 ... OK
   n_0.43 ... OK
   n_0.44 ... OK
   n_0.45 ... OK
   n_0.46 ... OK
   n_0.47 ... OK
   n_0.48 ... OK
   n_0.49 ... OK
   n_0.50 ... OK
   n_0.51 ... OK
   n_0.52 ... OK
   n_0.53 ... OK
   n_0.54 ... OK
   n_0.55 ... OK
   n_0.56 ... OK
   n_0.57 ... OK
   n_0.58 ... OK
   n_0.59 ... OK
   n_0.60 ... OK
   n_0.61 ... OK
   n_0.62 ... OK
   n_0.63 ... OK
   n_0.64 ... OK
   n_0.65 ... OK
   n_0.66 ... OK
   n_0.67 ... OK
   n_0.68 ... OK
   n_0.69 ... OK
   n_0.70 ... OK
   n_0.71 ... OK
   n_0.72 ... OK
   n_0.73 ... OK
   n_0.74 ... OK
   n_0.75 ... OK
   n_0.76 ... OK
   n_0.77 ..

In [6]:
# ============================================================
# 5. PIVOTAR AL FORMATO DESEADO
# ============================================================

df_long = pd.DataFrame(filas_long)

# Pivot: filas = FOTO+Clase, columnas = modelo_conf, valores = Conteo
df_pivot = df_long.pivot_table(
    index=['FOTO', 'Clase'],
    columns='modelo_conf',
    values='Conteo',
    aggfunc='first',
    fill_value=0
).reset_index()

# Aplanar multi-index de columnas si quedó
df_pivot.columns.name = None

# Ordenar columnas: FOTO, Clase, luego modelo_conf alfabéticamente
cols_base = ['FOTO', 'Clase']
cols_conf = [c for c in df_pivot.columns if c not in cols_base]
# Ordenar por tamaño y confianza para que quede lógico
def orden_col(c):
    if c in cols_base:
        return ('', 0, 0)
    partes = c.split('_')
    tam = partes[0]
    conf = float(partes[1])
    orden_tam = {'n':0, 's':1, 'm':2, 'l':3, 'x':4}
    return ('', orden_tam.get(tam, 99), conf)

cols_conf = sorted(cols_conf, key=orden_col)
df_final = df_pivot[cols_base + cols_conf]

print(f'Filas: {len(df_final)} | Columnas: {len(df_final.columns)}')
print('\n📋 Vista previa:')
print(df_final.head(10).to_string())

Filas: 1840 | Columnas: 357

📋 Vista previa:
  FOTO                   Clase  n_0.20  n_0.21  n_0.22  n_0.23  n_0.24  n_0.25  n_0.26  n_0.27  n_0.28  n_0.29  n_0.30  n_0.31  n_0.32  n_0.33  n_0.34  n_0.35  n_0.36  n_0.37  n_0.38  n_0.39  n_0.40  n_0.41  n_0.42  n_0.43  n_0.44  n_0.45  n_0.46  n_0.47  n_0.48  n_0.49  n_0.50  n_0.51  n_0.52  n_0.53  n_0.54  n_0.55  n_0.56  n_0.57  n_0.58  n_0.59  n_0.60  n_0.61  n_0.62  n_0.63  n_0.64  n_0.65  n_0.66  n_0.67  n_0.68  n_0.69  n_0.70  n_0.71  n_0.72  n_0.73  n_0.74  n_0.75  n_0.76  n_0.77  n_0.78  n_0.79  n_0.80  n_0.81  n_0.82  n_0.83  n_0.84  n_0.85  n_0.86  n_0.87  n_0.88  n_0.89  n_0.90  s_0.20  s_0.21  s_0.22  s_0.23  s_0.24  s_0.25  s_0.26  s_0.27  s_0.28  s_0.29  s_0.30  s_0.31  s_0.32  s_0.33  s_0.34  s_0.35  s_0.36  s_0.37  s_0.38  s_0.39  s_0.40  s_0.41  s_0.42  s_0.43  s_0.44  s_0.45  s_0.46  s_0.47  s_0.48  s_0.49  s_0.50  s_0.51  s_0.52  s_0.53  s_0.54  s_0.55  s_0.56  s_0.57  s_0.58  s_0.59  s_0.60  s_0.61  s_0.62  s_0.63  s_0

In [7]:
# ============================================================
# 6. GUARDAR Y DESCARGAR
# ============================================================

df_final.to_excel(RUTA_SALIDA, index=False)
print(f'💾 Guardado en: {RUTA_SALIDA}')

from google.colab import files
files.download(str(RUTA_SALIDA))

💾 Guardado en: /content/drive/MyDrive/conteo de aves marinas/conteos_pivotado.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>